# 📑 PageIndex — Vectorless RAG Crash Course
### Reasoning-based RAG with No Vector DB, No Chunking


---

## 🧠 What You'll Learn

| # | Topic |
|---|-------|
| 1 | Why Vector RAG fails on professional documents |
| 2 | How PageIndex builds a tree index from a PDF |
| 3 | LLM Tree Search — reasoning over structure |
| 4 | Full end-to-end Vectorless RAG pipeline |
| 5 | Expert-guided retrieval (domain knowledge injection) |
| 6 | Chat API — zero LLM setup |
| 7 | Self-hosted open-source option |

---

## 🔑 Key Concept

> **Traditional RAG** → chunk → embed → cosine similarity → retrieve  
> **PageIndex RAG** → build tree → LLM reasons over tree → retrieve exact sections

**The problem with vector RAG:**  
`Similarity ≠ Relevance`  
A chunk about "market conditions" may score higher than the actual answer section just because it shares more words with your query.


---
## 📦 Section 1: Install & Setup

**What we do here:**
- Install PageIndex SDK + OpenAI
- Load API keys from `.env`
- Initialize both clients

> 🔑 Get your **PageIndex API key** from: https://dash.pageindex.ai/api-keys  
> 🔑 Get your **GROQ API key** from: https://console.groq.com/keys


In [49]:
import os, json, time
from dotenv import load_dotenv
from groq import Groq


load_dotenv()

PAGEINDEX_API_KEY=os.getenv("PAGEINDEX_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")

print("PageIndex Key Loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("GROQ API Key Loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")

PageIndex Key Loaded: ✅
GROQ API Key Loaded: ✅


In [50]:
from groq import Groq
from pageindex import PageIndexClient
from langchain_groq import ChatGroq

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
groq_client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

print("✅ PageIndexClient is ready.")
print("✅ Groq client is ready.")

✅ PageIndexClient is ready.
✅ Groq client is ready.


---
## 🌲 Section 2: Upload & Index a PDF

**What happens here:**
1. Upload your PDF to the PageIndex cloud
2. PageIndex uses an LLM to read the document structure
3. Builds a hierarchical **tree index** (like a smart Table of Contents)
4. Returns a `doc_id` for all future operations

**Why NO chunking?**  
Instead of cutting the document into arbitrary 500-token pieces, PageIndex respects the document's natural section boundaries — chapters, sub-sections, paragraphs — as the author intended.


In [51]:
PDF_PATH = "/Users/nishantsingh04/Documents/RAG-Retrieval-Augmented-Generation/Data/pdf/machine_learning_foundations.pdf"

print(f"📥 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print("✅ Uploaded.")
print(f"📑 Document ID: {doc_id}")
print("   (Save this ID - you'll use it throughout the notebook.)")

📥 Uploading: /Users/nishantsingh04/Documents/RAG-Retrieval-Augmented-Generation/Data/pdf/machine_learning_foundations.pdf
✅ Uploaded.
📑 Document ID: pi-cmquuiie500lx01peofqofftq
   (Save this ID - you'll use it throughout the notebook.)


In [52]:
## PageIndex builds the tree asynchronously.
## For a 50-page PDF typically takes 30-90 seconds.

from grpc import Status


print("⏳ Building Tree Index...")
print("    (This runs once per document - The indiex is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")

    if status == "completed":
        print("\n✅ Tree Index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check you PDF format.")
        break

    time.sleep(5)

⏳ Building Tree Index...
    (This runs once per document - The indiex is cached for reuse)
   Status: processing
   Status: processing
   Status: processing
   Status: completed

✅ Tree Index ready!


---
## 🔍 Section 3: Inspect the Tree Structure

**What the tree looks like:**

```
Document
├── Introduction (pages 1-3)
│   └── Background (pages 1-2)
├── Financial Stability (pages 21-31)
│   ├── Monitoring Vulnerabilities (pages 22-28)
│   └── International Cooperation (pages 28-31)
└── Conclusion (pages 45-47)
```

Each node has:
- `node_id` — unique ID used during retrieval
- `title` — section heading
- `page_index` — page number in original PDF
- `text` — section summary (when `node_summary=True`)
- `nodes` — child sections (nested)

**This structure is what the LLM reasons over during retrieval.**


In [53]:
## Fetch the full tree

tree_result = pi_client.get_tree(doc_id, node_summary= True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 1

🌲 Raw tree (first node):
{
  "title": "Foundations of Machine Learning",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "This document serves as a technical reference manual and textbook titled 'Foundations of Machine Learning,' published in June 2026 as part of the Advanced Graduate Series in Artificial Intelligence. It provides a comprehensive curriculum spanning mathematical prerequisites, core learning paradigms, deep learning, sequence modeling, reinforcement learning, optimization techniques, MLOps, and ethical considerations in AI.",
  "text": "# Foundations of Machine Learning\n\nA Comprehensive Blueprint from Statistical Roots to Deep Architectures\n\n**Technical Reference Manual & Textbook**\n\nAdvanced Graduate Series in Artificial Intelligence\n\nPublished: June 2026\n\nFormat: Comprehensive Compendium\n\nTable of Contents\n\n|  Chapter 1: Introduction and Mathematical Prerequisites | 3  |\n| --- | --- |\n|  Chapter 2: Supervised Learni

In [54]:
## Pretty.print the full tree

def print_tree(nodes, indent = 0):
    """Recursively point tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + (" └─ " if indent > 0 else"")
        page = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']} (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Foundations of Machine Learning (p.1)
   └─ [0001] Chapter 1: Introduction and Mathematical Prerequisites (p.3)
     └─ [0002] 1.1 Linear Algebra Foundations (p.3)
     └─ [0003] 1.2 Probability and Statistical Inference (p.3)
   └─ [0004] Chapter 2: Supervised Learning Paradigms (p.5)
     └─ [0005] 2.1 Linear and Ridge Regression (p.5)
     └─ [0006] 2.2 Logistic Regression and Generalized Linear Models (p.5)
     └─ [0007] 2.3 Support Vector Machines (SVM) (p.5)
   └─ [0008] Chapter 3: Unsupervised Learning & Dimensionality Reduction (p.7)
   └─ [0009] Chapter 4: Generalization, Regularization, and Model Selection (p.8)
   └─ [0010] Chapter 5: Deep Learning Foundations (p.9)
     └─ [0011] 5.1 Multi-Layer Perceptrons (MLP) and Forward Propagation (p.9)
     └─ [0012] 5.2 Backpropagation and Automated Differentiation (p.9)
   └─ [0013] Chapter 6: Sequence Models and Natural Language Processing (p.11)
   └─ [0014] Chapter 7: Reinforcement Learning Fo

In [55]:
## Count Total nodes

def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrieval section section of the document")

🔢 Total nodes in tree: 20
   Each node = one retrieval section section of the document


---
## 🧠 Section 4: LLM Tree Search — The Core of PageIndex

**This is where PageIndex fundamentally differs from vector RAG.**

### Vector RAG retrieval:
```
query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
```
*Problem: finds what's similar, not what's relevant*

### PageIndex retrieval:
```
query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
```
*Advantage: LLM understands document structure, context, and intent*

**The LLM acts like a human expert scanning a Table of Contents.**


In [56]:
## LLM Tree search Function

def llm_tree_search(query: str, tree: list, model: str = "groq/compound-mini") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLm reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node_IDs)
    """

    ## Compress tree to save tokens - only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title": n["title"],
                "page": n.get("page_index", "?"),
                "summary": n.get("text", "")[:300] ## First 300 characters
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)

    prompt = f"""You are given a query and a documment's tree structure (like a Table of Contents).
    Your task: Identify which node IDs most likely contains the answer to the query.
    Think step-by step about which sections are relevant.property

    Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent = 2)}

Reply ONLY in this excat JSON format:
{{
    "thinking": "<your step-by-step reasoning>",
    "node_list": ["node_id1","node_id2"]
}}"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role":"user", "content": prompt}],
        response_format={"type":"json_object"}
    )

    return json.loads(response.choices[0].message.content)

In [57]:
## Test with a sample query

query = "What is the topic covered in the Machine Learning Foundation?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What is the topic covered in the Machine Learning Foundation?

🧠 LLM Reasoning:
The query asks for the topic covered in the 'Machine Learning Foundation'. The most direct match in the hierarchy is the top‑level node whose title is 'Foundations of Machine Learning' (node_id 0000). Its summary describes the overall subject of the work – a comprehensive blueprint of machine learning foundations. No other node explicitly mentions a 'Machine Learning Foundation' phrase; the other nodes refer to specific sub‑foundations (e.g., Deep Learning Foundations). Therefore the answer is most likely located in node 0000.

🎯 Selected Node IDs: ['0000']


---
## ⚙️ Section 5: Full End-to-End RAG Pipeline

**3 steps:**
1. **Tree Search** → LLM picks relevant `node_ids`
2. **Retrieve** → Fetch the actual section content from those nodes  
3. **Generate** → LLM writes a grounded answer with page citations

**What makes this better than vector RAG:**
- Retrieved content has titles + page numbers (traceable)
- LLM can cite exactly *which section* the answer comes from
- No hallucination from irrelevant chunks


In [58]:
## Helper: Find nodes by IDs

def find_nodes_by_ids(tree: list, target_ids: list)-> list:
    """Recursively walk the tree and collect the nodes matching the target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found


In [70]:
## generate answer from retrieved nodes

def generate_answer(query: str, nodes: list, model: str = "llama-3.1-8b-instant") -> str:
    """
    Take Retrieved nodes as context and generate a grounded answer.
     Instructs LLm to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ NO Relevant sections sound in the document."
    
    ## Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n--\n\n".join(context_parts)

    prompt = f"""You are an expert document analyst.
    Answer the question using ONLY provided context.
    For every claim you make cite the section title and page number in parentheses.
    Be concise and precise.
    
    Question: {query}
    
    Context:
    {context}
    
    Answer:"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role":"user", "content": prompt}]
        )   
    
    return response.choices[0].message.content
    


In [71]:
## The complete VertorLess RAG function

from networkx import find_asteroidal_triple


def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG PipeLine:
    
    Step 1: LLM Tree Serach -> Finds relevant node_ids.
    Step 2: Node Retrieval -> Fetches Section content.
    Step 3: Answer Generation -> Produces cited answer
    """

    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")

    # Step 1: Tree Search
    search_result = llm_tree_search(query, tree)
    node_ids = search_result.get("node_list", [])

    if verbose:
         print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
         print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)

    if verbose:
        print(f"📄 Section Found: {[n['title'] for n in nodes]}")

    # Step 3: Generate Answer
    answer = generate_answer(query, nodes)

    if verbose:
        print(f"\n📝Answer: \n{answer}")

    return answer

        

In [72]:
## Run the Full PipeLine

answer = vectorless_rag(
    query ="What are the topic covered in Machine Learning Foundation?",
    tree = pageindex_tree
)

🔍 Query: What are the topic covered in Machine Learning Foundation?

🧠 Reasoning: The query asks for the topics covered in the Machine Learning Foundation. In the document tree, the foundational topics are represented by the main chapter nodes under the root 'Foundations of Machine...
🎯 Retrieved node IDs: ['0001', '0004', '0008', '0009', '0010', '0013', '0014', '0015', '0018', '0019']
📄 Section Found: ['Chapter 1: Introduction and Mathematical Prerequisites', 'Chapter 2: Supervised Learning Paradigms', 'Chapter 3: Unsupervised Learning & Dimensionality Reduction', 'Chapter 4: Generalization, Regularization, and Model Selection', 'Chapter 5: Deep Learning Foundations', 'Chapter 6: Sequence Models and Natural Language Processing', 'Chapter 7: Reinforcement Learning Foundations', 'Chapter 8: Optimization Techniques and Gradient Descent Variants', 'Chapter 9: Production Machine Learning (MLOps)', 'Chapter 10: Ethical AI, Bias, and Interpretability']

📝Answer: 
The topics covered in Machin

In [73]:
## Test with multiple queries
test_queries = [
    "What are the types of Machine Learning?",
    "What is Generalization?",
    "Summarize the syllabus of Supervised Learning?",
]

for q in test_queries:
    print()
    ans = vectorless_rag(q, pageindex_tree, verbose=False)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}...")
    print("-" * 55)



Q: What are the types of Machine Learning?
A: There are primarily two categories to machine learning: Supervised Learning and Unsupervised Learning, along with another distinct category: Reinforcement Learning.

1. **Supervised Learning** (Chapter 2: Supervised Learning Paradigms, Page 5) involves systems where training datasets possess explici...
-------------------------------------------------------



RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01ktbvvmgrfd7r4e6anftkbzzd` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 9525, Requested 2776. Please try again in 1.505s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'compound', 'code': 'rate_limit_exceeded'}}

---
## 🎓 Section 6: Expert-Guided Retrieval

**The killer feature no one talks about.**

With vector RAG, injecting domain expertise requires **fine-tuning the embedding model** — expensive and time-consuming.

With PageIndex, you just **add rules to the prompt**:

```
"If the query mentions EBITDA → prioritize the MD&A section"
"If the query is about risks  → check Part I, Item 1A"
```

This makes PageIndex instantly adaptable to any domain — finance, legal, medical, technical — without any model training.


In [67]:
# ── Define domain expert rules ───────────────────────────────────────────────
# These are routing rules that tell the LLM WHERE to look for specific queries.
# Think of it as encoding a senior analyst's institutional knowledge.


In [68]:
FINANCIAL_EXPERT_RULES = """
Expert routing rules for financial documents (10-K, annual reports):
- EBITDA, profitability queries    → MD&A section (Management Discussion & Analysis)
- Liquidity, cash flow queries     → Cash Flow Statement + liquidity footnotes
- Risk factor queries              → Part I, Item 1A (Risk Factors)  
- Revenue breakdown queries        → Segment reporting or Item 7
- Forward-looking / strategy       → CEO letter, Outlook, Strategy section
- Debt, credit, leverage queries   → Balance Sheet + debt footnotes
- Regulatory / compliance queries  → Legal Proceedings or regulatory filings
"""

print("✅ Expert rules defined")
print("   These get injected into the retrieval prompt at query time.")

✅ Expert rules defined
   These get injected into the retrieval prompt at query time.


In [69]:
# ── Expert Routing Rules — Advanced Route of Learning AI ─────────────────────
# Krish Naik Academy | 21 Modules | 38 Sections | 481 Topics
FINANCIAL_EXPERT_RULES = """
Route queries to the correct module using these rules:
 
M1  Neural Network Refresher   → backprop, activations, optimizers, PyTorch basics
M2  Hardware                   → GPU, TPU, Apple Silicon, compute infrastructure
M3  Transformers 101           → attention, self-attention, encoder-decoder, MHA
M4  Tokenization               → BPE, WordPiece, SentencePiece, Byte Latent Transformers
M5  Finetuning Architectures   → hands-on BERT/GPT/T5 finetuning, Hugging Face
M6  KV Cache & Attention       → KV cache, Flash Attention, MQA, GQA, RoPE, vLLM
M7  Scaling Laws               → Kaplan, Chinchilla, compute-optimal training
M8  Mixture of Experts         → MoE, sparse computation, Mixture of Depths
M9  Modern LLM Finetuning      → LoRA, QLoRA, SFT, DPO, PPO, RLHF, GRPO, ORPO,
                                  quantization, TRL, Unsloth, synthetic data,
                                  reasoning models, evaluation, deployment
M10 SLM                        → small language models, pruning, when SLM vs LLM
M11 Knowledge Distillation     → student-teacher, soft labels, DistilBERT, DeepSeek-R1
M12 Hybrid Architectures       → Mamba, RWKV, SSMs, Jamba, Nemotron, beyond Transformers
M13 Vision Foundations         → ViT, patch embeddings, CLIP, SigLIP, DINOv2
M14 Visual Language Models     → VLM architecture, aligner, multimodal reasoning
M15 Stable Diffusion & DiT     → DDPM, latent diffusion, FLUX.1, ControlNet, DreamBooth
M16 Embedding Models           → dense, sparse, binary, Matryoshka, MRL, fine-tuning
M17 RAG                        → chunking, BM25, ColBERT, hybrid RAG, rerankers,
                                  self/corrective/adaptive/agentic RAG, Graph RAG,
                                  multi-modal RAG, ColPali, RAG security
M18 Context Engineering        → prompt vs context engineering, memory architecture,
                                  context compression, KV cache, agent context lifecycle
M19 DSPy                       → signatures, modules, MIPROv2, self-optimizing RAG
M20 Agents                     → ReAct, MCP, LangGraph, CrewAI, browser agents,
                                  A2A, guardrails, observability, evaluation
M21 RL                         → PPO, GRPO, DAPO, GSPO, CISPO, reward models,
                                  RLHF vs RLVR, policy gradient, DeepSeek-R1 training
 
Cross-cutting rules:
- "learning path / where to start"     → M1 → M2 → M3 in order
- "production / deployment / serving"  → M9 (quantization) + M20 (agents)
- "fine-tuning vs RAG"                 → M9 + M17 + M18
- "multimodal / vision + language"     → M13 + M14 + M17 (multi-modal RAG)
- "reasoning models / test-time RL"    → M9 (reasoning) + M21 (GRPO/DAPO)
"""

In [74]:
## Expert Guided tree Search

def llm_tree_search_with_expert(
    query: str,
    tree: list,
    expert_rules: str,
    model: str = "qwen/qwen3-32b"
) -> dict:
    """
    Same as llm_tree_search() but with domain expert rules injected.
    The LLM uses these rules to guide its reasoning.
    """
    
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {"node_id": n["node_id"], "title": n["title"],
                     "page": n.get("page_index", "?"),
                     "summary": n.get("text", "")[:150]}
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    prompt = f"""You are a domain expert analyzing a document.
Find all node IDs that most likely contain the answer to the query.
Use the expert routing rules below to guide your reasoning.

Query: {query}

Document Tree:
{json.dumps(compress(tree), indent=2)}

Expert Routing Rules (follow these carefully):
{expert_rules}

Reply ONLY in this JSON format:
{{
  "thinking": "<your reasoning, referencing the expert rules>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

In [75]:
## Test expert-guided retrieval
query = "Details of the Machine Learning?"

print(f"🔍 Query: {query}\n")

# Without expert rules
print("── Without Expert Rules ──")
basic   = llm_tree_search(query, pageindex_tree)
print("Nodes:", basic.get("node_list"))

print()

# With expert rules
print("── With Expert Rules ──")
guided  = llm_tree_search_with_expert(query, pageindex_tree, FINANCIAL_EXPERT_RULES)
print("Nodes:", guided.get("node_list"))
print("Reasoning:", guided.get("thinking", "")[:300])

🔍 Query: Details of the Machine Learning?

── Without Expert Rules ──
Nodes: ['0000', '0001']

── With Expert Rules ──
Nodes: ['0000', '0001', '0004', '0010']
Reasoning: The query seeks general details about Machine Learning. The document's root node (0000) provides the foundational overview. Since the query is broad, we route to introductory chapters first (M1 rule: 'learning path / where to start' → M1 → M2 → M3 in order). Nodes 0001 (mathematical foundations), 00


In [76]:
def expert_rag(query: str, tree: list, rules: str) -> str:
    """Expert-guided end-to-end RAG pipeline."""
    result  = llm_tree_search_with_expert(query, tree, rules)
    nodes   = find_nodes_by_ids(tree, result.get("node_list", []))
    return generate_answer(query, nodes)

# Run it
answer = expert_rag(
    query="Details of the syllabus of Machine Learning",
    tree=pageindex_tree,
    rules=FINANCIAL_EXPERT_RULES
)
print(answer)

The syllabus of Machine Learning as per the provided context consists of the following topics:

1. Mathematical Prerequisites (Chapter 1: Introduction and Mathematical Prerequisites, page 3) - Covers topics like data-driven statistical inference, functional mappings, and mapping conditions.
2. Supervised Learning Paradigms (Chapter 2: Supervised Learning Paradigms, page 5) - Discusses regression and classification mapping their loss spaces and optimizing structural trajectories.
3. Unsupervised Learning & Dimensionality Reduction (Chapter 3: Unsupervised Learning & Dimensionality Reduction, page 7) - Examines clustering methods (K-Means) and dimensionality reduction techniques (PCA).
4. Generalization, Regularization, and Model Selection (Chapter 4: Generalization, Regularization, and Model Selection, page 9) - Covers bias-variance tradeoff and techniques to prevent overfitting.
5. Deep Learning Foundations (Chapter 5: Deep Learning Foundations, page 9) - Introduces deep learning and i

---
## 💬 Section 7: Chat API — Zero LLM Setup

**When to use this:**
- You don't want to manage OpenAI API calls yourself
- You want a quick Q&A interface over your document
- You're building a chat product and want PageIndex to handle everything

PageIndex provides its own LLM — you just pass a question and `doc_id`.


In [77]:
## Single question with Chat API
## No OpenAI key needed — PageIndex runs the LLM internally

question = "What are the key findings in this document?"

response = pi_client.chat_completions(
    messages=[{"role": "user", "content": question}],
    doc_id=doc_id
)

answer = response["choices"][0]["message"]["content"]
print("💬 Chat API Answer:")
print(answer)

💬 Chat API Answer:
Here's a structured summary of the key findings across this textbook's 10 chapters:

---

## 📘 Key Findings: *Foundations of Machine Learning*

### 1. Mathematical Prerequisites
- ML is fundamentally rooted in **linear algebra** (vectors, matrices, SVD) and **Bayesian probability**, where learning is framed as posterior inference: `P(θ|D) ∝ P(D|θ) · P(θ)`.

### 2. Supervised Learning
- **Linear/Ridge Regression** minimizes MSE loss; **Logistic Regression** uses the sigmoid function with Binary Cross-Entropy for classification.
- **SVMs** achieve classification by maximizing the geometric margin to support vectors, using hinge loss with L2 regularization.

### 3. Unsupervised Learning & Dimensionality Reduction
- **K-Means** clusters data by minimizing within-cluster variance. **PCA** reduces dimensions via SVD. Non-linear alternatives include **t-SNE** (visualization) and **UMAP** (topological structure).

### 4. Generalization & Regularization
- The **bias-variance 

In [78]:
## Multi-turn conversation 
## Keep the full message history for context across turns

conversation_history = []

def chat_with_doc(user_message: str, doc_id: str) -> str:
    """Chat with a document, maintaining conversation history."""
    global conversation_history
    
    conversation_history.append({"role": "user", "content": user_message})
    
    response = pi_client.chat_completions(
        messages=conversation_history,
        doc_id=doc_id
    )
    
    assistant_reply = response["choices"][0]["message"]["content"]
    conversation_history.append({"role": "assistant", "content": assistant_reply})
    
    return assistant_reply


# Simulate a 3-turn conversation
questions = [
    "What were the main revenue sources last year?",
    "How does that compare to the year before?",
    "What factors drove that change?"
]

for q in questions:
    print(f"\n👤 User: {q}")
    reply = chat_with_doc(q, doc_id)
    print(f"🤖 Assistant: {reply[:400]}...")
    print("-" * 55)


👤 User: What were the main revenue sources last year?
🤖 Assistant: The document you have — **machine_learning_foundations_2.pdf** — is a machine learning textbook covering topics like mathematical foundations, deep learning, and MLOps. It doesn't contain any financial or revenue information.

If you're looking for revenue data, you'd need to provide a financial report or similar document. Would you like to ask something about the ML textbook instead?...
-------------------------------------------------------

👤 User: How does that compare to the year before?
🤖 Assistant: As mentioned, the document available (**machine_learning_foundations_2.pdf**) is a machine learning textbook — it contains no financial or revenue data for any year, so a year-over-year comparison isn't possible from this source.

If you'd like revenue comparisons, please upload the relevant financial reports. Otherwise, I'm happy to help with anything related to the ML textbook!...
-----------------------------------

---
## 🛠️ Section 8: Self-Hosted Open Source Option

**Use this when:**
- You don't want to send documents to any cloud
- You need full data privacy / on-prem deployment
- You want to inspect or customize the tree-building logic

The open-source repo at https://github.com/VectifyAI/PageIndex lets you run the entire pipeline locally using your own OpenAI key.

**What the CLI does:**
1. Reads your PDF
2. Detects existing Table of Contents (if any)
3. Uses GPT-4o to build the hierarchical tree
4. Saves a `document_name_pageindex.json` alongside your PDF


In [1]:
# ── Create .env for self-hosted mode ─────────────────────────────────────────
# The local runner uses CHATGPT_API_KEY (not OPENAI_API_KEY)

import os
openai_key = os.getenv("OPENAI_API_KEY", "your_key_here")

with open(".env", "w") as f:
    f.write(f"CHATGPT_API_KEY={openai_key}\n")

print("✅ .env created for self-hosted mode")

✅ .env created for self-hosted mode


In [ ]:


# ── Run PageIndex locally on a PDF ───────────────────────────────────────────
# Optional parameters you can customize:
#   --model                  OpenAI model (default: gpt-4o-2024-11-20)
#   --toc-check-pages        Pages to scan for existing TOC (default: 20)
#   --max-pages-per-node     Max pages per tree node (default: 10)
#   --if-add-node-summary    Include summaries in output (yes/no)

PDF_PATH = "/path/to/your/document.pdf"   # ← change this

!python run_pageindex.py \
    --pdf_path {PDF_PATH} \
    --model gpt-4o-2024-11-20 \
    --toc-check-pages 20 \
    --max-pages-per-node 10 \
    --if-add-node-summary yes



python: can't open file '/Users/nishantsingh04/Documents/RAG-Retrieval-Augmented-Generation/run_page_index.py': [Errno 2] No such file or directory


In [5]:
# ── Load locally generated tree ──────────────────────────────────────────────
# Output is saved as: <your_pdf_name>_pageindex.json

import json

TREE_JSON_PATH = "/path/to/your/document_pageindex.json"  # ← change this

with open(TREE_JSON_PATH, "r") as f:
    local_tree = json.load(f)

print(f"🌲 Local tree loaded: {count_nodes(local_tree)} total nodes")
print_tree(local_tree)

FileNotFoundError: [Errno 2] No such file or directory: '/path/to/your/document_pageindex.json'

In [ ]:


# ── Run the same RAG pipeline on the local tree ──────────────────────────────
# Everything from Sections 4–6 works identically with local trees

query  = "Summarize the executive summary section."
answer = vectorless_rag(query, local_tree)



---
## 📊 Section 9: Vector RAG vs PageIndex — Side-by-Side

### Architecture Comparison

| Aspect | Traditional Vector RAG | PageIndex (Vectorless RAG) |
|--------|------------------------|---------------------------|
| **Document prep** | Chunk into fixed pieces | Build hierarchical tree |
| **Indexing** | Embed each chunk | LLM reads structure |
| **Storage** | Vector database | JSON file |
| **Query processing** | Embed query → ANN search | LLM reasons over tree |
| **What's retrieved** | Flat anonymous chunks | Named sections + page refs |
| **Explainability** | ❌ Opaque similarity score | ✅ Traceable reasoning |
| **Domain expertise** | ❌ Needs embedding fine-tune | ✅ Add rules to prompt |
| **Infrastructure** | Pinecone / FAISS / ChromaDB | No vector DB needed |
| **Best for** | Short, diverse documents | Long, structured documents |
| **FinanceBench accuracy** | ~80% | **98.7%** |

### When to use which

**Use Vector RAG when:**
- Documents are short and varied (FAQs, product descriptions)
- Semantic paraphrase matching is important  
- You need sub-second retrieval on millions of documents

**Use PageIndex when:**
- Documents are long and professionally structured (reports, manuals, legal docs)
- You need traceable, cited answers
- Domain expertise should guide retrieval
- You want to avoid vector DB infrastructure


In [79]:
## Quick comparison demo 
## Show how the same query retrieves differently

print("=" * 55)
print("VECTOR RAG approach (conceptual):")
print("=" * 55)
print("""
query_vec = embed_model.encode("What are EBITDA risks?")
chunks    = vector_db.similarity_search(query_vec, k=5)

# Returns: 5 text fragments ranked by cosine distance
# Problem: may return "market risk" chunks, not EBITDA section
# No page numbers, no section context
""")

print("=" * 55)
print("PAGEINDEX approach (actual):")
print("=" * 55)
print("""
result = llm_tree_search("What are EBITDA risks?", tree)

# Returns: node IDs like ["0007", "0012"]
# LLM reasoning: "EBITDA is discussed in MD&A section (node 0007)
#                 and footnotes in Financial Statements (node 0012)"
# Full traceability — section title + page number
""")

VECTOR RAG approach (conceptual):

query_vec = embed_model.encode("What are EBITDA risks?")
chunks    = vector_db.similarity_search(query_vec, k=5)

# Returns: 5 text fragments ranked by cosine distance
# Problem: may return "market risk" chunks, not EBITDA section
# No page numbers, no section context

PAGEINDEX approach (actual):

result = llm_tree_search("What are EBITDA risks?", tree)

# Returns: node IDs like ["0007", "0012"]
# LLM reasoning: "EBITDA is discussed in MD&A section (node 0007)
#                 and footnotes in Financial Statements (node 0012)"
# Full traceability — section title + page number



---
## 🧹 Section 10: Cleanup

Delete documents from the PageIndex cloud when you're done  
to keep your storage clean.


In [ ]:
## Delete document from cloud
## WARNING: This permanently deletes the tree index.
## Comment this out if you want to reuse the doc_id later.

## pi_client.delete_document(doc_id)
## print(f"🗑️ Deleted document: {doc_id}")
print("ℹ️ Deletion commented out — uncomment when you're done with this doc_id")


---
## ✅ Summary

You've now built a complete **Vectorless RAG** system with PageIndex.

### What you built:

1. **`llm_tree_search()`** — LLM reasons over document tree to find relevant nodes
2. **`find_nodes_by_ids()`** — Retrieve actual section content from tree
3. **`generate_answer()`** — LLM produces cited, grounded answers
4. **`vectorless_rag()`** — Full pipeline combining all 3 steps
5. **`expert_rag()`** — Domain-guided retrieval without any fine-tuning
6. **Chat API** — Zero-setup document Q&A

### Key takeaways:

- `Similarity ≠ Relevance` — the fundamental flaw of vector search
- Tree-based reasoning gives you **traceable**, **accurate**, **explainable** retrieval
- Domain expertise injection is just **prompt engineering** — no model training needed
- 98.7% on FinanceBench vs ~80% for vector RAG

---

### 🔗 Resources
- GitHub: https://github.com/VectifyAI/PageIndex
- Docs: https://docs.pageindex.ai
- Chat Platform: https://chat.pageindex.ai
- Blog: https://pageindex.ai/blog/pageindex-intro

---
